In [1]:
import json
from pprint import pprint
import torch
import os
import ast
import pandas as pd
from openai import OpenAI
import concurrent.futures



In [2]:
# read csv file

df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,llm_errors,num_groups,agreement,text_length,llm_result_final,issues,llm_a1_agreement,llm_a2_agreement,a1_a2_agreement,llm_final_agreement
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,4.0,TRUE,110,"[\n {\n ""Chest pain"": [\n ""non stop c...",[],TRUE,TRUE,TRUE,TRUE
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,NaN,TRUE,18,No temporal information found,[],TRUE,TRUE,TRUE,TRUE
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,NaN,2.0,FALSE,13,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","['llm formatting issue', 'incorrect grouping',...",FALSE,FALSE,TRUE,FALSE
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,NaN,2.0,0,260,"[\n {\n ""Vaccination site pain"": [\n ...",[],TRUE,0,0,TRUE
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,2.0,1,118,"[\n {\n ""Syncope"": [\n ""patient faint...",[],1,1,1,1


In [3]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [4]:
# Gold timeline = human-corrected version when one exists, otherwise the accepted
# o4-mini result. A report has temporal progression unless that gold timeline is
# "No temporal information found". (The earlier filter also tested
# llm_result_formatted on its own, which dropped the 10 reports where o4-mini saw
# no progression but the annotators corrected it to a multi-stage timeline,
# giving 3156 instead of the 3166 MedTempo-T reports stated in the paper.)
data['gold_timeline'] = data['final_corrected_version'].where(
    data['final_corrected_version'].notna(), data['llm_result_formatted'])
data['no_temporal_info'] = data['gold_timeline'].astype(str).str.lower().str.contains('no temporal information')
data['no_temporal_info'].value_counts()
# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()
# we have to remove final_result value 2 from data_with_temporal_info
data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
data_with_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3794 entries, 0 to 5998
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3794 non-null   int64 
 1   symptom_text             3794 non-null   object
 2   symptom_list             3794 non-null   object
 3   llm_result               3794 non-null   object
 4   llm_result_formatted     3794 non-null   object
 5   final_corrected_version  654 non-null    object
 6   final_result             3794 non-null   int64 
 7   no_temporal_info         3794 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 240.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   ob

/tmp/ipykernel_486180/3745764098.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['no_temporal_info'] = data.apply(lambda row:


In [5]:
# find data with no temporal info
data_no_temporal_info = data[data['no_temporal_info'] == True]
data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

data_no_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2191 entries, 1 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 2191 non-null   int64 
 1   symptom_text             2191 non-null   object
 2   symptom_list             2191 non-null   object
 3   llm_result               2191 non-null   object
 4   llm_result_formatted     2191 non-null   object
 5   final_corrected_version  238 non-null    object
 6   final_result             2191 non-null   int64 
 7   no_temporal_info         2191 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 139.1+ KB


In [6]:
# take 50 each from data_with_temporal_info and data_no_temporal_info
data_sampled_with_temporal_info = data_with_temporal_info.sample(n=200, random_state=67)
data_sampled_no_temporal_info = data_no_temporal_info.sample(n=100, random_state=67)
# concatenate the two dataframes
data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
data_sampled.info()
# save to csv
# data_sampled.to_csv('sample_40.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 300 entries, 133 to 2284
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 300 non-null    int64 
 1   symptom_text             300 non-null    object
 2   symptom_list             300 non-null    object
 3   llm_result               300 non-null    object
 4   llm_result_formatted     300 non-null    object
 5   final_corrected_version  47 non-null     object
 6   final_result             300 non-null    int64 
 7   no_temporal_info         300 non-null    bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 19.0+ KB


In [7]:
# now lets take 40 random samples from data_with_temporal_info with random_state=2
sampled_data = data_sampled_no_temporal_info 
sampled_data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 5115 to 2284
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 100 non-null    int64 
 1   symptom_text             100 non-null    object
 2   symptom_list             100 non-null    object
 3   llm_result               100 non-null    object
 4   llm_result_formatted     100 non-null    object
 5   final_corrected_version  12 non-null     object
 6   final_result             100 non-null    int64 
 7   no_temporal_info         100 non-null    bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 6.3+ KB


In [8]:
eg = data_with_temporal_info[['vaers_id','symptom_text', 'symptom_list', 'llm_result_formatted','final_corrected_version','final_result']]
#eg.to_csv('openai_gold_with_temporal.csv', index=False)

In [9]:
eg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   object
 2   symptom_list             3156 non-null   object
 3   llm_result_formatted     3156 non-null   object
 4   final_corrected_version  651 non-null    object
 5   final_result             3156 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 172.6+ KB


In [10]:
# # assume temporal information exists for all
# example = """
# [
#   {"Erythema": ["redness in neck"]},
#   {
#     "Pain in extremity": ["sore arm"],
#     "Pruritus": ["itchy feeling"]
#   },
#   {"Swelling": ["mild arm swelling"]}
# ]
# """

# input_template_mention = """
# Ignore previous conversations.

# Clinical Notes: {text}

# Task:
#     - Use the adverse effect list below:
#      {suggest}
#    - Reorder the list based on the sequence implied in the clinical notes.
#    - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
#    - Do **not skip any terms** from the list — include all terms in the output.
#    - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

# Important Rules:
# - Mention each symptom **only once** — at the first time it appears or becomes relevant.
# - Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
# - If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
# - Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.

# Finally, return your answer **only** in valid JSON format —
# using a **list of dictionaries** to represent temporal progression and grouping, as shown below:
# {example}
# """

In [11]:
# 5 shot prompt
example = """
[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]
"""
few_shot_examples = """

Example 1 
Clinical Notes: I was at the vaccination venue when I received my first Pfizer injection on 4/1/2021 at 10:41. About 15 minutes after the injection, I started to feel extremely hot and like I was suffocating. My face started to swell, I had difficulty swallowing, and I felt dizzy. The paramedics who were on site read my blood pressure and oxygen levels, which were fine. They monitored me for the next 4 hours until I felt well enough to go home. I took Benadryl and Tylenol until 4/3/2021. I felt dizzy until about 4/4/2021. I felt lethargic and weak until about 4/5/2021, and I then felt fine afterwards.
Adverse effect list: ['Asthenia', 'Dizziness', 'Dysphagia', 'Feeling hot', 'Lethargy', 'Suffocation feeling', 'Swelling face']
Output:
[
  {
    "Feeling hot": [
      "feel extremely hot"
    ],
    "Suffocation feeling": [
      "like I was suffocating"
    ],
    "Swelling face": [
      "My face started to swell"
    ],
    "Dysphagia": [
      "difficulty swallowing"
    ],
    "Dizziness": [
      "felt dizzy"
    ]
  },
  {
    "Asthenia": [
      "weak"
    ],
    "Lethargy": [
      "felt lethargic"
    ]
  }
]

Example 2 
Clinical Notes: 8/2/2021: presented to ED with CC of shortness of breath.  Symptoms began 4 days prior to presentation, patient began experiencing shortness of breath & cough productive of white sputum, body aches, and fever/chills. Patient endorses intermittent headaches, nausea, and diarrhea for the past 2 days and intermittent left flank pain he attributes to his recurrent nephrolithiasis.  Diagnosed with acute hypoxic resp failure 2/2 COVID-19 Note: patient previously vaccinated with Pfizer COVID-19 vaccine on 3/3/21 and 4/13/21. 8/12/2021: patient discharged
Adverse effect list: ['Acute respiratory failure', 'Chills', 'Condition aggravated', 'Diarrhoea', 'Dyspnoea', 'Flank pain', 'Headache', 'Nausea', 'Nephrolithiasis', 'Pain', 'Productive cough', 'Pyrexia']
Output:
[
  {
    "Chills": [
      "chills"
    ],
    "Dyspnoea": [
      "shortness of breath"
    ],
    "Pain": [
      "body aches"
    ],
    "Productive cough": [
      "cough productive of white sputum"
    ],
    "Pyrexia": [
      "fever"
    ]
  },
  {
    "Diarrhoea": [
      "diarrhea"
    ],
    "Flank pain": [
      "intermittent left flank pain"
    ],
    "Headache": [
      "intermittent headaches"
    ],
    "Nausea": [
      "nausea"
    ],
    "Nephrolithiasis": [
      "recurrent nephrolithiasis"
    ]
  },
  {
    "Acute respiratory failure": [
      "acute hypoxic resp failure"
    ]
  },
  {
    "Condition aggravated": [
      "none"
    ]
  }
]

Example 3
Clinical Notes: Patient presented to the ED with 1 day history of chest and abdominal pain on 2/3/2022. States the pain lasts for around 30 minutes. The first 10 minutes described as 10/10 and then subsides to 3/10 over the course of 30 minutes. Pain located in the center of chest that radiates down to abdomen. Pain does not radiate to arms, back, or neck.  Does not feel the pain is related to movement or eating but it does get better when passing gas or burping. He vomited once yesterday morning, no diarrhea. Endorses pleuritic chest pain. He received his COVID-19 immunization 4 days ago, and has had headache and body ache since for which he has taken ibuprofen regularly.
Adverse effect list: ['Abdominal pain', 'Blood creatine phosphokinase increased', 'Blood lactate dehydrogenase increased', 'C-reactive protein increased', 'Chest pain', 'Electrocardiogram ST segment abnormal', 'Fibrin D dimer increased', 'Headache', 'Pain', 'Pleuritic pain', 'Troponin increased', 'Vomiting']
Output:
[
  {
    "Headache": [
      "headache"
    ],
    "Pain": [
      "body ache"
    ]
  },
  {
    "Vomiting": [
      "vomited once yesterday morning"
    ],
    "Chest pain": [
      "chest pain"
    ],
    "Abdominal pain": [
      "abdominal pain"
    ],
    "Pleuritic pain": [
      "pleuritic chest pain"
    ]
  },
  {
    "Fibrin D dimer increased": [
      "none"
    ],
    "Blood lactate dehydrogenase increased": [
      "none"
    ],
    "Blood creatine phosphokinase increased": [
      "none"
    ],
    "Electrocardiogram ST segment abnormal": [
      "none"
    ],
    "Troponin increased": [
      "none"
    ],
    "C-reactive protein increased": [
      "none"
    ]
  }
]

Example 4
Clinical Notes: "Patient who had recently been vaccinated approached me and stated that he did not feel well." "Complaining of weakness and dizziness after COVID Vaccine." "Pale and notably diaphoretic" "Vital signs assessed and found to be bradycardia and hypotensive in seated position. EKG completed and transport to medical facility via medic ambulance."  11:43 BP 90/31, HR48 (weak, regular), RR 16 normal and SpO2 98% room air. 11:46 BP 86/39, HR52, SpO2 98% 11:55 BP 125/68, HR73, SpO2 100%  12lead performed to show brady-sinus with no indications of heartblock. IV fluids, 100mL LR via 20g in LAC.
Adverse effect list: ['Asthenia', 'Bradycardia', 'Dizziness', 'Hyperhidrosis', 'Hypotension', 'Malaise', 'Pallor']
Output:
[
  {
    "Malaise": [
      "did not feel well"
    ],
    "Asthenia": [
      "weakness"
    ],
    "Dizziness": [
      "dizziness"
    ],
    "Pallor": [
      "Pale"
    ],
    "Hyperhidrosis": [
      "notably diaphoretic"
    ]
  },
  {
    "Bradycardia": [
      "bradycardia"
    ],
    "Hypotension": [
      "hypotensive"
    ]
  }
]

Example 5 
Clinical Notes: 6 hours after J&J shot, began shivering uncontrollably and immediately went to bed. Was in bed with flu-like symptoms for 24 hours.  Took Advil  Day 2 and 3 after shot, had a mild headache. Took ibuprofen   Day 4 to present, noticed a red, raised, somewhat itchy ?bubble? patch of skin on the side of my nose, which my mom said looks like Hives. I also have red patches of slightly itchy skin under my eyes. Took Benadryl but doesn?t seem to be helping
Adverse effect list: ['Chills', 'Headache', 'Influenza like illness', 'Rash', 'Rash erythematous', 'Rash papular', 'Rash pruritic', 'Urticaria']
Output:
[
  {
    "Chills": [
      "shivering uncontrollably"
    ]
  },
  {
    "Influenza like illness": [
      "flu-like symptoms"
    ]
  },
  {
    "Headache": [
      "mild headache"
    ]
  },
  {
    "Rash erythematous": [
      "red patches of slightly itchy skin under my eyes"
    ],
    "Rash papular": [
      "red, raised, somewhat itchy ?bubble? patch of skin on the side of my nose"
    ],
    "Rash pruritic": [
      "red patches of slightly itchy skin under my eyes"
    ],
    "Urticaria": [
      "red, raised, somewhat itchy ?bubble? patch of skin on the side of my nose"
    ],
    "Rash": [
      "red patches of slightly itchy skin under my eyes"
    ]
  }
]
"""
input_template_mention = """

Ignore previous conversations.

Task:
    - Use the adverse effect list below:
     {suggest}
   - Reorder the list based on the sequence implied in the clinical notes.
   - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
   - Do **not skip any terms** from the list — include all terms in the output.
   - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom **only once** — at the first time it appears or becomes relevant.
- Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
- Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.


Return format:
- Return **only** valid JSON.
- Output must be a **list of dictionaries** (each dictionary = one temporal block), like:
{example}

Assume temporal information exists for all examples below.

Examples:

{few_shot_examples}

Now solve the next case:
Clinical Notes: {text}
Adverse findings list: {suggest}
Output:"""

In [12]:
import os
import openai 
import time 
import random
# chatgpt results
openai_api_key = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=openai_api_key)


def llm(prompt, retries=5, delay=2, backoff=2):
    for attempt in range(retries):
        try:
            kwargs = dict(
            model="gpt-4.1",
            input=prompt,
            temperature= 0,
            # max_output_tokens=512
            )
            response = client.responses.create(**kwargs)
            return response.output_text
        except openai.RateLimitError:
            print(f"RateLimitError. Retrying in {delay} sec...")
        except openai.Timeout:
            print(f"Timeout. Retrying in {delay} sec...")
        except Exception as e:
            print(f"Other error: {str(e)}. Retrying in {delay} sec...")
        time.sleep(delay)
        delay *= backoff + random.uniform(0, 1)  # jitter to avoid spikes
    return "Error: Failed after retries"
    
def clean_and_parse_json(text):
    try:
        if isinstance(text, str):
            cleaned = text.replace('\n', '')
            return json.loads(cleaned)
    except json.JSONDecodeError:
        return None

    return None

def validate_output_symptoms(row):
    if isinstance(row['llm_result_cleaned'], list):
        output_symptoms = [key for dic in row['llm_result_cleaned'] for key in dic.keys()]
        input_symptoms = ast.literal_eval(row['symptom_list'])
        return set(output_symptoms) == set(input_symptoms)
    return False

def get_ordered_symptoms_flatten(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.extend(list(sym_dict.keys()))
    return symptoms

def get_ordered_symptoms(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.append(list(sym_dict.keys()))
    return symptoms

def call_llm(row):
    
    prompt = input_template_mention.format(  
        suggest=row['symptom_list'],
        example=example,
        few_shot_examples=few_shot_examples,
        text=row['symptom_text']
        
    )
    return llm(prompt)

In [13]:

batch_size = 500
for i in range(0, len(eg), batch_size):
    print(f"Processing batch {i} to {i+batch_size}...")

    batch = eg.iloc[i:i+batch_size]
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(call_llm, [row for _, row in batch.iterrows()]))

    eg.loc[batch.index, 'llm_result'] = results
    eg.to_csv(f"openai_baseline_5shot_with_temporal_part_{i}.csv", index=False)  # Save checkpoint

   


Processing batch 0 to 500...


/tmp/ipykernel_486180/3003524652.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eg.loc[batch.index, 'llm_result'] = results


Processing batch 500 to 1000...
Processing batch 1000 to 1500...
Processing batch 1500 to 2000...
Processing batch 2000 to 2500...
Processing batch 2500 to 3000...
Processing batch 3000 to 3500...


In [14]:
import re
def parse_json_best_effort(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (text.find("{"), text.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(text) - 1, start, -1):
            if text[end] in "}]":
                chunk = text[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None

In [15]:
res_lst = []
for row in eg.iterrows():
    res = parse_json_best_effort(row[1]['llm_result'])
    res_lst.append(res)

# store res_lst in reslts_df
eg['final_result_cleaned'] = res_lst
eg.head()



,vaers_id,symptom_text,symptom_list,llm_result_formatted,final_corrected_version,final_result,llm_result,final_result_cleaned
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,"[\n {\n ""Chest pain"": [\n ""non stop c...",[{'Chest pain': ['non stop chest pain that was...
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Immediate post-injection reaction...",[{'Immediate post-injection reaction': ['Muscl...
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,"[\n {\n ""Vaccination site pain"": [\n ...",[{'Vaccination site pain': ['injection site pa...
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,"[\n {\n ""Syncope"": [\n ""patient faint...","[{'Syncope': ['patient fainted'], 'Fall': ['th..."
5,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,"[\n {\n ""Tunnel vision"": [\n ""\""tunne...","[{'Tunnel vision': ['""tunnel vision""'], 'Dizzi..."


In [16]:
eg.to_csv(f"openai_baseline_5shot_with_temporal_final.csv", index=False)  # Save checkpoint


In [17]:
# eg['ordered_symptoms'] = eg['llm_result_cleaned'].apply(get_ordered_symptoms)

In [18]:
# eg['ordered_symptoms_flatten'] = eg['llm_result_cleaned'].apply(get_ordered_symptoms_flatten)

In [19]:
# eg[eg['is_symptom_list_correct'] == False]

In [20]:
# sampled_data['is_symptom_list_correct'].value_counts()

In [21]:
# sampled_data['llm_result'][sampled_data['is_symptom_list_correct'] == False].value_counts()

In [22]:
# def check_length(row):
#     ordered_symptoms_list = len(row['ordered_symptoms_flatten'])
#     symptom_list = len(ast.literal_eval(row['symptom_list']))
#     return ordered_symptoms_list == symptom_list

# sampled_data['is_same_length'] = sampled_data.apply(check_length, axis=1)

In [23]:
# sampled_data.to_excel('data/sampled_annotated_data.xlsx', index=False)